# Liar's Dice: All-Out Agent War Tournament

This notebook runs a comprehensive round-robin tournament between all available agents with statistically significant sample sizes (200 games per agent pair) and balanced starting positions (role switching). It computes win ratios, confidence intervals, and visualizes head-to-head matchups.

## Tournament Design

- **Pairings**: All ordered pairs of agents (includes self-play)
- **Games per pair**: 200 full matches (multi-round elimination-style)
- **Role balance**: 100 games as player 0 (first to move), 100 as player 1
- **Sample size**: Large enough for reliable statistical inference (~95% CI)
- **Matchup duration**: Each match ends when a player reaches 0 dice (full elimination)

## Output

- CSV files with detailed game summaries, tournament pairwise results, and agent-level statistics
- Visualizations: heatmap of head-to-head win rates, bar chart with confidence intervals
- Statistical summary: win ratios, CI bounds, role-specific performance

## 1. Environment Setup and Imports

In [49]:
import os
import sys
import pathlib
import datetime
import csv
import itertools
import time
from collections import defaultdict
from typing import Tuple
from scipy import stats

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Import ipywidgets for interactive filtering
try:
    import ipywidgets as widgets
    from IPython.display import display
    _WIDGETS_AVAILABLE = True
except ImportError:
    _WIDGETS_AVAILABLE = False
    print("Warning: ipywidgets not available. Installing with: pip install ipywidgets")

# Discover project root without hard-coding user paths

def find_repo_root() -> pathlib.Path:
    repo_name = "Adaptive-Strategy-Learning-in-Liar-s-Dice"

    def is_repo(path: pathlib.Path) -> bool:
        return (path / "liars_dice").is_dir() and (path / "scripts").is_dir()

    # 1) Honor manual override
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root:
        candidate = pathlib.Path(env_root).expanduser().resolve()
        if is_repo(candidate):
            return candidate

    # 2) Walk up from current working directory
    start = pathlib.Path.cwd().resolve()
    for candidate in [start] + list(start.parents):
        if is_repo(candidate):
            return candidate

    # 3) Try direct repo folder under home (no hard-coded subpaths)
    home_candidate = pathlib.Path.home() / repo_name
    if is_repo(home_candidate):
        return home_candidate.resolve()

    raise RuntimeError("Could not locate project root. Set PROJECT_ROOT or start the notebook inside the repo.")

notebook_dir = pathlib.Path.cwd().resolve()
project_root = find_repo_root()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Switch working directory to project root so relative paths resolve correctly
try:
    os.chdir(str(project_root))
except Exception as e:
    print(f"Warning: failed to change CWD to project root: {e}")

print(f"Notebook directory: {notebook_dir}")
print(f"Project root: {project_root}")
print(f"Current working directory: {pathlib.Path.cwd()}")
print("Python path updated.")
print(f"Interactive widgets available: {_WIDGETS_AVAILABLE}")


Notebook directory: C:\Users\shaha\Work\Personal\AIGT\Adaptive-Strategy-Learning-in-Liar-s-Dice
Project root: C:\Users\shaha\Work\Personal\AIGT\Adaptive-Strategy-Learning-in-Liar-s-Dice
Current working directory: C:\Users\shaha\Work\Personal\AIGT\Adaptive-Strategy-Learning-in-Liar-s-Dice
Python path updated.
Interactive widgets available: True


## 2. Load Tournament Utilities

Import key utilities from `run_tournament_full_game.py`. These include the core `run_full_match()` function and tournament infrastructure.

In [28]:
# Import core game and agent modules
from liars_dice.agents import AGENT_MAP
from liars_dice.core.config import GameConfig
from liars_dice.persistence import csv_io

print(f"Available agents: {sorted(list(AGENT_MAP.keys()))}")
print(f"Total agent types: {len(AGENT_MAP)}")

Available agents: ['aggressive', 'alternator', 'bayesian', 'bluffing', 'chaotic', 'chaotic_safe', 'chaotic_unsafe', 'conservative', 'cycleface', 'maxcount', 'maxraise', 'minraise', 'mirror', 'nash_cfr', 'onesarewild', 'parity', 'probability_maxraise', 'probability_minraise', 'random', 'random_aggressive', 'random_cautious', 'random_facefixed', 'random_facerandom', 'randomface', 'randomthreshold', 'rl_ppo', 'safeface', 'thresholdliar']
Total agent types: 28


In [29]:
# Import tournament functions from the script using an absolute path for robustness
rtfg_path = project_root / "scripts" / "run_tournament_full_game.py"
if not rtfg_path.exists():
    raise FileNotFoundError(f"run_tournament_full_game.py not found at {rtfg_path}")

import importlib.util
spec = importlib.util.spec_from_file_location("run_tournament_full_game", str(rtfg_path))
run_tournament_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_tournament_module)

generate_game_id = run_tournament_module.generate_game_id
run_full_match = run_tournament_module.run_full_match

print(f"Tournament functions imported from {rtfg_path}")

Tournament functions imported from C:\Users\shaha\Work\Personal\AIGT\Adaptive-Strategy-Learning-in-Liar-s-Dice\scripts\run_tournament_full_game.py


## 3. Configure Tournament Parameters

Set up the tournament configuration: which agents to compete, number of games per pairing, and output directory.

In [30]:
# Tournament Configuration
AGENTS = sorted(list(AGENT_MAP.keys()))  # Use all available agents
GAMES_PER_PAIR = 200  # Total games per ordered pair (100 as P0, 100 as P1)
DATA_DIR = 'data'  # Use existing data directory

# Create output directory if it doesn't exist
os.makedirs(DATA_DIR, exist_ok=True)

print(f"Tournament Configuration:")
print(f"  Agents: {AGENTS}")
print(f"  Games per ordered pair: {GAMES_PER_PAIR}")
print(f"  Total agent pairs: {len(AGENTS) ** 2}")
print(f"  Total games: {len(AGENTS) ** 2 * GAMES_PER_PAIR}")
print(f"  Output directory: {DATA_DIR} (appending to existing files)")

Tournament Configuration:
  Agents: ['aggressive', 'alternator', 'bayesian', 'bluffing', 'chaotic', 'chaotic_safe', 'chaotic_unsafe', 'conservative', 'cycleface', 'maxcount', 'maxraise', 'minraise', 'mirror', 'nash_cfr', 'onesarewild', 'parity', 'probability_maxraise', 'probability_minraise', 'random', 'random_aggressive', 'random_cautious', 'random_facefixed', 'random_facerandom', 'randomface', 'randomthreshold', 'rl_ppo', 'safeface', 'thresholdliar']
  Games per ordered pair: 200
  Total agent pairs: 784
  Total games: 156800
  Output directory: data (appending to existing files)


In [32]:
from pathlib import Path
from liars_dice.agents.reinforcement_agent.config import TRAINING_CONFIG

# Quick sanity checks for model path resolution
model_path = TRAINING_CONFIG["model_save_path"]
path_obj = Path(model_path)
if path_obj.suffix != ".zip":
    path_obj = path_obj.with_suffix(".zip")
if not path_obj.is_absolute():
    # In notebooks __file__ is not set; use project_root discovered in cell 1
    path_obj = (Path(project_root) / path_obj).resolve()

print("Model path (resolved):", path_obj)
print("Exists?", path_obj.exists())


Model path (resolved): C:\Users\shaha\Work\Personal\AIGT\Adaptive-Strategy-Learning-in-Liar-s-Dice\liars_dice\agents\weights\ppo_model.zip
Exists? True


## 4. Run Round-Robin War Tournament

Execute the full tournament with balanced role assignments. Each pair plays 200 games total:
- First 100 games: Agent A as player 0 (first to move), Agent B as player 1
- Last 100 games: Agent B as player 0 (first to move), Agent A as player 1

**Note**: This cell may take a significant amount of time to complete depending on the number of agents and game complexity.

In [33]:
# Initialize tracking structures
cfg = GameConfig()
agent_stats = defaultdict(lambda: defaultdict(int))
tournament_rows = []
summary_header = csv_io.get_summary_header()
trajectory_header = csv_io.get_trajectory_header()

# Prepare file paths
summary_csv = os.path.join(DATA_DIR, 'game_summary.csv')
trajectory_csv = os.path.join(DATA_DIR, 'game_trajectory.csv')
tournament_csv = os.path.join(DATA_DIR, 'tournament_summary.csv')
agent_csv = os.path.join(DATA_DIR, 'agent_tournament_stats.csv')

# Ensure output directory exists (robust even if earlier cells weren't run)
os.makedirs(os.path.dirname(summary_csv), exist_ok=True)

# Generate all ordered pairs (includes self-play for baseline)
pairs = list(itertools.product(AGENTS, AGENTS))
total_games = len(pairs) * GAMES_PER_PAIR
timestamp_base = datetime.datetime.now(datetime.timezone.utc).isoformat()
start_time = time.time()

print(f"🎮 Starting tournament with {len(pairs)} agent pairs ({total_games:,} total games)")
print()

for (a0_key, a1_key) in tqdm(pairs, desc="Agent Pairs", unit="pair"):
    a0_cls = AGENT_MAP[a0_key]
    a1_cls = AGENT_MAP[a1_key]
    pair_wins = {'a0': 0, 'a1': 0}
    pair_steps = 0
    pair_bids = 0
    pair_calls = 0
    pair_bluffs = 0
    pair_errors = 0

    for i in tqdm(range(GAMES_PER_PAIR), desc=f"{a0_key} vs {a1_key}", leave=False, unit="game"):
        ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
        game_id = generate_game_id(a0_cls, a1_cls, f"{ts}_{i}")
        
        # Run the full match
        summary_row, trajectory_rows = run_full_match(a0_cls, a1_cls, cfg, i, game_id, ts)
        
        # Persist results
        csv_io.append_row_to_csv(summary_row, summary_csv, summary_header)
        if trajectory_rows:
            csv_io.append_rows_to_csv(trajectory_rows, trajectory_csv, trajectory_header)

        # Update statistics
        winner = summary_row.get('winner')
        if winner is None:
            pair_errors += 1
        else:
            if winner == 0:
                pair_wins['a0'] += 1
                agent_stats[a0_key]['wins'] += 1
                agent_stats[a0_key]['wins_as_starter'] += 1
            elif winner == 1:
                pair_wins['a1'] += 1
                agent_stats[a1_key]['wins'] += 1
                agent_stats[a1_key]['wins_as_second'] += 1
            agent_stats[a0_key]['games'] += 1
            agent_stats[a1_key]['games'] += 1

        pair_steps += int(summary_row.get('steps') or 0)
        pair_bids += int(summary_row.get('bids') or 0)
        pair_calls += int(summary_row.get('calls') or 0)
        pair_bluffs += int(summary_row.get('bluffs_called') or 0)
        if summary_row.get('error'):
            pair_errors += 1

    # Record pairwise tournament results
    games_played = GAMES_PER_PAIR
    row = {
        'timestamp': timestamp_base,
        'agent0': a0_key,
        'agent1': a1_key,
        'games': games_played,
        'wins_agent0': pair_wins['a0'],
        'wins_agent1': pair_wins['a1'],
        'avg_steps': (pair_steps / games_played) if games_played > 0 else 0.0,
        'total_bids': pair_bids,
        'total_calls': pair_calls,
        'total_bluffs_called': pair_bluffs,
        'errors': pair_errors,
    }
    tournament_rows.append(row)

# Write tournament pairwise results
tour_header = ['timestamp', 'agent0', 'agent1', 'games', 'wins_agent0', 'wins_agent1', 'avg_steps', 'total_bids', 'total_calls', 'total_bluffs_called', 'errors']
with open(tournament_csv, 'a', newline='', encoding='utf-8') as f:
    import csv
    writer = csv.DictWriter(f, fieldnames=tour_header)
    if not os.path.getsize(tournament_csv) > 0:
        writer.writeheader()
    for r in tournament_rows:
        writer.writerow(r)

# Write per-agent statistics
agent_rows = []
agent_header = ['agent', 'games', 'wins', 'win_percent', 'wins_as_starter', 'wins_as_second']
for agent in sorted(AGENTS):
    g = agent_stats[agent].get('games', 0)
    w = agent_stats[agent].get('wins', 0)
    win_percent = (w / g * 100.0) if g > 0 else 0.0
    agent_rows.append({
        'agent': agent,
        'games': g,
        'wins': w,
        'win_percent': f"{win_percent:.3f}",
        'wins_as_starter': agent_stats[agent].get('wins_as_starter', 0),
        'wins_as_second': agent_stats[agent].get('wins_as_second', 0),
    })

with open(agent_csv, 'a', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=agent_header)
    if not os.path.getsize(agent_csv) > 0:
        writer.writeheader()
    for r in agent_rows:
        writer.writerow(r)

from math import isfinite

# Avoid divide by zero if tournament aborted early
avg_time = (time.time() - start_time) / total_games if total_games else float('nan')
print()
print(f"✅ Tournament completed. Total games scheduled: {total_games:,}")
if isfinite(avg_time):
    print(f"⏱️  Approx. average time per game: {avg_time:.3f}s")
print(f"💾 Results saved to {DATA_DIR}/")

🎮 Starting tournament with 784 agent pairs (156,800 total games)



Agent Pairs:   0%|          | 0/784 [00:00<?, ?pair/s]

Agent Pairs: 100%|██████████| 784/784 [3:33:57<00:00, 16.37s/pair]    


✅ Tournament completed. Total games scheduled: 156,800
⏱️  Approx. average time per game: 0.082s
💾 Results saved to data/


## 5. Load Results into DataFrames

Read the generated CSV files and perform basic sanity checks.

In [34]:
# Load tournament results
tournament_df = pd.read_csv(tournament_csv)
agent_df = pd.read_csv(agent_csv)
summary_df = pd.read_csv(summary_csv)

print(f"Tournament DataFrame shape: {tournament_df.shape}")
print(f"Agent stats DataFrame shape: {agent_df.shape}")
print(f"Game summary DataFrame shape: {summary_df.shape}")
print()
print("Tournament data (first 10 rows):")
print(tournament_df.head(10))
print()
print("Agent statistics:")
print(agent_df)

Tournament DataFrame shape: (784, 11)
Agent stats DataFrame shape: (28, 6)
Game summary DataFrame shape: (158534, 15)

Tournament data (first 10 rows):
                          timestamp      agent0          agent1  games  \
0  2026-01-22T10:27:53.484235+00:00  aggressive      aggressive    200   
1  2026-01-22T10:27:53.484235+00:00  aggressive      alternator    200   
2  2026-01-22T10:27:53.484235+00:00  aggressive        bayesian    200   
3  2026-01-22T10:27:53.484235+00:00  aggressive        bluffing    200   
4  2026-01-22T10:27:53.484235+00:00  aggressive         chaotic    200   
5  2026-01-22T10:27:53.484235+00:00  aggressive    chaotic_safe    200   
6  2026-01-22T10:27:53.484235+00:00  aggressive  chaotic_unsafe    200   
7  2026-01-22T10:27:53.484235+00:00  aggressive    conservative    200   
8  2026-01-22T10:27:53.484235+00:00  aggressive       cycleface    200   
9  2026-01-22T10:27:53.484235+00:00  aggressive        maxcount    200   

   wins_agent0  wins_agent1  avg_

C:\Users\shaha\AppData\Local\Temp\ipykernel_5828\1900536453.py:4: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  summary_df = pd.read_csv(summary_csv)


## 6. Compute Win Ratios and Role-Balanced Win Percentages

Calculate overall win percentage for each agent, as well as role-specific performance (starter vs second player).

The **role-balanced win ratio** is the mean of the two role-specific win rates:
$$\hat{p}_{\text{balanced}} = \frac{1}{2}\left(\frac{W_{\text{starter}}}{N_{\text{starter}}} + \frac{W_{\text{second}}}{N_{\text{second}}}\right)$$

In [35]:
# Compute win statistics
agent_df['win_percent'] = pd.to_numeric(agent_df['win_percent'])
agent_df['total_games_per_role'] = agent_df['games'] // 2  # Assuming balanced roles

# Compute role-specific win percentages
agent_df['starter_win_percent'] = (agent_df['wins_as_starter'] / (agent_df['total_games_per_role'])) * 100.0
agent_df['second_win_percent'] = (agent_df['wins_as_second'] / (agent_df['total_games_per_role'])) * 100.0
agent_df['balanced_win_percent'] = (agent_df['starter_win_percent'] + agent_df['second_win_percent']) / 2.0

print("Agent win statistics (with role breakdown):")
print(agent_df[['agent', 'games', 'wins', 'win_percent', 'starter_win_percent', 'second_win_percent', 'balanced_win_percent']].to_string())
print()

# Summary statistics
print("\nSummary Statistics:")
print(f"Overall avg win rate: {agent_df['win_percent'].mean():.2f}%")
print(f"Overall avg starter win rate: {agent_df['starter_win_percent'].mean():.2f}%")
print(f"Overall avg second player win rate: {agent_df['second_win_percent'].mean():.2f}%")
print(f"\nBest overall performer: {agent_df.loc[agent_df['win_percent'].idxmax(), 'agent']} ({agent_df['win_percent'].max():.2f}%)")
print(f"Worst overall performer: {agent_df.loc[agent_df['win_percent'].idxmin(), 'agent']} ({agent_df['win_percent'].min():.2f}%)")

Agent win statistics (with role breakdown):
                   agent  games   wins  win_percent  starter_win_percent  second_win_percent  balanced_win_percent
0             aggressive  11200   6241       55.723            59.589286           51.857143             55.723214
1             alternator  11200   6860       61.250            50.482143           72.017857             61.250000
2               bayesian  11200   9688       86.500           100.000000           73.000000             86.500000
3               bluffing  11200   6557       58.545            62.857143           54.232143             58.544643
4                chaotic  11200   3300       29.464            34.392857           24.535714             29.464286
5           chaotic_safe  11200   3257       29.080            33.339286           24.821429             29.080357
6         chaotic_unsafe  11200   3279       29.277            34.267857           24.285714             29.276786
7           conservative  11200   95

## 7. Quick Start: Dynamic Agent Filtering

Use the interactive widgets below to easily filter which agents to analyze. No more manual copy-paste needed!

**Features:**
- ☑️ Checkboxes to select/deselect agents
- 🎚️ Slider for minimum win rate threshold  
- 🔄 Real-time filtering with visual feedback
- 📊 All downstream analyses update automatically

In [ ]:
# Create interactive filtering widget interface
print("="*80)
print("INTERACTIVE AGENT FILTERING")
print("="*80)

# Get all agents sorted
all_agents_sorted = sorted(list(AGENT_MAP.keys()))

# Create checkboxes for each agent
agent_checkboxes = {agent: widgets.Checkbox(value=True, description=agent) for agent in all_agents_sorted}

# Create slider for minimum win rate threshold
win_rate_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=100,
    step=5,
    description='Min Win %:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

# Create a button to apply filters
apply_button = widgets.Button(description='Apply Filters', button_style='info', tooltip='Click to apply selected filters')

# Create reset button
reset_button = widgets.Button(description='Reset All', button_style='warning', tooltip='Click to reset filters')

# Output area for messages
output_area = widgets.Output()

def apply_filters(button):
    """Apply the selected filters."""
    with output_area:
        output_area.clear_output()
        
        # Collect selected agents
        selected_agents = [agent for agent, checkbox in agent_checkboxes.items() if checkbox.value]
        
        # Collect excluded agents
        excluded = [agent for agent, checkbox in agent_checkboxes.items() if not checkbox.value]
        
        # Get threshold
        threshold = win_rate_slider.value
        
        # Update global variables
        globals()['EXCLUDED_AGENTS'] = excluded
        globals()['MIN_WIN_RATE_THRESHOLD'] = threshold
        
        print(f"✓ Filters applied!")
        print(f"  Selected agents: {len(selected_agents)}")
        print(f"  Excluded agents: {len(excluded)}")
        if excluded:
            print(f"    {', '.join(excluded)}")
        print(f"  Min win rate threshold: {threshold}%")
        print(f"\\n→ Run the filtered analysis cells below to see results.")

def reset_filters(button):
    """Reset all filters to default."""
    with output_area:
        output_area.clear_output()
        
        # Reset all checkboxes to True
        for checkbox in agent_checkboxes.values():
            checkbox.value = True
        
        # Reset slider
        win_rate_slider.value = 0
        
        # Update globals
        globals()['EXCLUDED_AGENTS'] = []
        globals()['MIN_WIN_RATE_THRESHOLD'] = 0
        
        print("✓ All filters reset to default (show all agents)")

# Attach event handlers
apply_button.on_click(apply_filters)
reset_button.on_click(reset_filters)

# Create the UI layout
print("\\nSelect agents to INCLUDE in analysis (uncheck to exclude):")
print()

# Create a scrollable container for checkboxes
checkbox_vbox = widgets.VBox(list(agent_checkboxes.values()), layout=widgets.Layout(
    border='1px solid #ddd',
    padding='10px',
    height='200px',
    overflow_y='auto'
))

# Create the filter control panel
control_panel = widgets.VBox([
    widgets.HTML("<b>Filtering Controls</b>"),
    checkbox_vbox,
    widgets.HTML("<br>"),
    win_rate_slider,
    widgets.HBox([apply_button, reset_button]),
    output_area
])

display(control_panel)

# Initialize with default values
EXCLUDED_AGENTS = []
MIN_WIN_RATE_THRESHOLD = 0

print("\\n✓ Widget interface loaded. Select agents and click 'Apply Filters' to update analysis.")

INTERACTIVE AGENT FILTERING
\nSelect agents to INCLUDE in analysis (uncheck to exclude):



\n✓ Widget interface loaded. Select agents and click 'Apply Filters' to update analysis.


**How to use the filtering interface:**

1. **Select agents**: Uncheck the boxes next to any agents you want to exclude from the analysis
2. **Set minimum win rate**: Use the slider to show only agents with at least that win percentage
3. **Click "Apply Filters"**: This will update the filter configuration
4. **Run filtered cells**: Execute the cells below to see updated visualizations and statistics
5. **Click "Reset All"**: Quickly restore all agents and reset the threshold to 0%

The filtered analysis will automatically update to show only your selected agents.

### Filtered Agent Statistics

Compute 95% Confidence Intervals for the selected agents.

Using the normal approximation (Wilson score interval would be more accurate, but this is sufficient for large n):
$$CI = \hat{p} \pm 1.96 \sqrt{\frac{\hat{p}(1-\hat{p})}{n}}$$

This provides a range where we expect the true win probability to lie with 95% confidence.

In [42]:
print("\n" + "="*100)
print("FILTERED TOURNAMENT STATISTICS")
print("="*100)
print(f"\nApplied Filters:")
print(f"  Excluded agents: {EXCLUDED_AGENTS if EXCLUDED_AGENTS else 'None'}")
print(f"  Min win rate threshold: {MIN_WIN_RATE_THRESHOLD}%")
print()

# Create filtered dataset
filtered_agent_df = agent_df[
    (~agent_df['agent'].isin(EXCLUDED_AGENTS)) & 
    (agent_df['win_percent'] >= MIN_WIN_RATE_THRESHOLD)
].copy()

filtered_agents = sorted(filtered_agent_df['agent'].unique())

print(f"Results:")
print(f"  Total agents: {len(AGENTS)}")
print(f"  Filtered agents: {len(filtered_agents)}")
print(f"  Excluded count: {len(AGENTS) - len(filtered_agents)}")
print()

if len(filtered_agent_df) == 0:
    print("⚠ No agents match the current filter criteria. Try adjusting filters above.")
else:
    print(filtered_agent_df[['agent', 'games', 'wins', 'win_percent', 'ci_lower', 'ci_upper',
                             'starter_win_percent', 'second_win_percent']].sort_values('win_percent', ascending=False).to_string(index=False))
    print("="*100)
    print(f"\nFiltered Summary:")
    print(f"  Average win rate: {filtered_agent_df['win_percent'].mean():.2f}%")
    print(f"  Median win rate: {filtered_agent_df['win_percent'].median():.2f}%")
    print(f"  Std dev: {filtered_agent_df['win_percent'].std():.2f}%")
    print(f"  Min: {filtered_agent_df['win_percent'].min():.2f}%")
    print(f"  Max: {filtered_agent_df['win_percent'].max():.2f}%")


FILTERED TOURNAMENT STATISTICS

Applied Filters:
  Excluded agents: ['random', 'random_aggressive', 'random_cautious', 'random_facefixed', 'random_facerandom', 'randomface', 'randomthreshold']
  Min win rate threshold: 0%

Results:
  Total agents: 28
  Filtered agents: 21
  Excluded count: 7

               agent  games  wins  win_percent  ci_lower  ci_upper  starter_win_percent  second_win_percent
              rl_ppo  11200 10346       92.375 91.883477 92.866523            99.964286           84.785714
            bayesian  11200  9688       86.500 85.867119 87.132881           100.000000           73.000000
        conservative  11200  9522       85.018 84.356876 85.678838            84.750000           85.285714
probability_minraise  11200  9255       82.634 81.932349 83.335508            86.750000           78.517857
       thresholdliar  11200  8648       77.214 76.437454 77.991118            77.357143           77.071429
            maxcount  11200  7725       68.973 68.116461 

### Filtered Head-to-Head Win Matrix

Heatmap showing only selected agents' matchups.

In [47]:
# Filter tournament data for selected agents
filtered_tournament_df = tournament_df[
    (tournament_df['agent0'].isin(filtered_agents)) & 
    (tournament_df['agent1'].isin(filtered_agents))
].copy()

# Create filtered head-to-head matrix
filtered_tournament_df['agent0_win_percent'] = (filtered_tournament_df['wins_agent0'] / filtered_tournament_df['games']) * 100.0

filtered_h2h_matrix = filtered_tournament_df.pivot_table(
    index='agent0',
    columns='agent1',
    values='agent0_win_percent',
    aggfunc='first'
)

# Reorder by filtered agents
filtered_h2h_matrix = filtered_h2h_matrix.reindex(filtered_agents).reindex(filtered_agents, axis=1)

print(f"\nFiltered Head-to-Head Matrix ({len(filtered_agents)} x {len(filtered_agents)}):")
print(filtered_h2h_matrix.round(1))
print()

# Visualize filtered heatmap
if len(filtered_agents) > 0:
    fig_size = max(8, len(filtered_agents) * 1.2)
    plt.figure(figsize=(fig_size, fig_size))
    sns.heatmap(
        filtered_h2h_matrix,
        annot=True,
        fmt='.1f',
        cmap='RdYlGn',
        center=50,
        vmin=0,
        vmax=100,
        cbar_kws={'label': 'Win % (as player 0)'},
        square=True,
        linewidths=0.5
    )
    plt.title(f'Filtered Agent War: Head-to-Head Win Rates\n({len(filtered_agents)} agents, Player 0 Perspective)')
    plt.xlabel('Opponent (Player 1)')
    plt.ylabel('Agent (Player 0)')
    plt.tight_layout()
    filtered_h2h_path = os.path.join(DATA_DIR, 'filtered_head_to_head_heatmap.png')
    plt.savefig(filtered_h2h_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Head-to-head heatmap saved to: {filtered_h2h_path}")
else:
    print("No agents to display after filtering.")


Filtered Head-to-Head Matrix (21 x 21):
agent1                aggressive  alternator  bayesian  bluffing  chaotic  \
agent0                                                                      
aggressive                 100.0         0.0       0.0     100.0     94.0   
alternator                 100.0       100.0      58.5       0.5     95.5   
bayesian                   100.0       100.0     100.0     100.0    100.0   
bluffing                     0.0        71.5      65.5      92.5    100.0   
chaotic                      5.5         7.0       0.0      19.5     60.0   
chaotic_safe                 8.0         7.5       0.0      14.0     49.5   
chaotic_unsafe               5.0         6.5       0.0      13.5     62.5   
conservative               100.0         0.0       0.0     100.0    100.0   
cycleface                    0.0       100.0       0.0       0.5     99.0   
maxcount                   100.0         0.0     100.0      92.5    100.0   
maxraise                     0.0   

C:\Users\shaha\AppData\Local\Temp\ipykernel_5828\86278241.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Filtered Bar Chart with Confidence Intervals

Win rates for selected agents only, sorted by performance.

In [44]:
# Sort filtered data by win percentage
sorted_filtered_df = filtered_agent_df.sort_values('win_percent', ascending=False).reset_index(drop=True)

if len(sorted_filtered_df) > 0:
    # Calculate error bars
    errors = np.array([
        sorted_filtered_df['win_percent'].values - sorted_filtered_df['ci_lower'].values,
        sorted_filtered_df['ci_upper'].values - sorted_filtered_df['win_percent'].values
    ])

    # Create filtered bar chart
    fig, ax = plt.subplots(figsize=(max(10, len(sorted_filtered_df) * 1.2), 6))
    bars = ax.bar(
        range(len(sorted_filtered_df)),
        sorted_filtered_df['win_percent'].values,
        color='steelblue',
        alpha=0.7,
        edgecolor='navy',
        linewidth=1.5
    )

    # Add error bars for 95% CI
    ax.errorbar(
        range(len(sorted_filtered_df)),
        sorted_filtered_df['win_percent'].values,
        yerr=errors,
        fmt='none',
        color='red',
        alpha=0.6,
        capsize=5,
        linewidth=2,
        label='95% CI'
    )

    # Add value labels on bars
    for i, (bar, val) in enumerate(zip(bars, sorted_filtered_df['win_percent'].values)):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{val:.1f}%',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_ylabel('Win Percentage (%)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Agent', fontsize=12, fontweight='bold')
    ax.set_title(f'Filtered Agent War: Win Rates ({len(sorted_filtered_df)} agents)\n95% Confidence Intervals', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(sorted_filtered_df)))
    ax.set_xticklabels(sorted_filtered_df['agent'].values, rotation=45, ha='right')
    ax.set_ylim(0, 105)
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper right')

    plt.tight_layout()
    filtered_win_chart_path = os.path.join(DATA_DIR, 'filtered_overall_win_rates.png')
    plt.savefig(filtered_win_chart_path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"✓ Filtered win rate chart saved to: {filtered_win_chart_path}")
else:
    print("No agents to display after filtering.")

✓ Filtered win rate chart saved to: data\filtered_overall_win_rates.png


C:\Users\shaha\AppData\Local\Temp\ipykernel_5828\936114392.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Filtered Head-to-Head Comparison Table

Detailed pairwise statistics between selected agents, showing who wins more often when each plays as player 0.

In [45]:
# Create detailed head-to-head comparison table
if len(filtered_tournament_df) > 0:
    h2h_comparison = filtered_tournament_df[[
        'agent0', 'agent1', 'games', 'wins_agent0', 'wins_agent1', 'agent0_win_percent', 'avg_steps'
    ]].copy()
    
    h2h_comparison.columns = ['Player 0', 'Player 1', 'Games', 'P0 Wins', 'P1 Wins', 'P0 Win %', 'Avg Steps']
    h2h_comparison = h2h_comparison.sort_values('P0 Win %', ascending=False)
    
    print("\n" + "="*100)
    print("FILTERED HEAD-TO-HEAD DETAILED COMPARISON")
    print("="*100)
    print(h2h_comparison.to_string(index=False))
    print("="*100)
else:
    print("No filtered matchups to display.")


FILTERED HEAD-TO-HEAD DETAILED COMPARISON
            Player 0             Player 1  Games  P0 Wins  P1 Wins  P0 Win %  Avg Steps
       thresholdliar             safeface    200      200        0     100.0     71.440
          aggressive           aggressive    200      200        0     100.0     15.000
       thresholdliar probability_maxraise    200      200        0     100.0     15.000
            safeface probability_maxraise    200      200        0     100.0     15.000
            safeface               parity    200      200        0     100.0     34.000
            safeface             maxraise    200      200        0     100.0     15.000
            safeface       chaotic_unsafe    200      200        0     100.0     18.900
            safeface         chaotic_safe    200      200        0     100.0     17.580
            safeface              chaotic    200      200        0     100.0     18.135
          alternator             maxraise    200      200        0     100.0 

In [48]:
# Top-5 agents: find worst adversaries by merged and by role-specific losses
import math

# Identify top 5 agents by overall win percentage
_top5_agents = agent_df.sort_values('win_percent', ascending=False).head(5)['agent'].tolist()

# Helper to compute role-specific and merged win rates for `agent` vs `opponent`
def _pair_stats(agent: str, opponent: str, tdf: pd.DataFrame):
    row_ab = tdf[(tdf['agent0'] == agent) & (tdf['agent1'] == opponent)]
    row_ba = tdf[(tdf['agent0'] == opponent) & (tdf['agent1'] == agent)]

    def _get(row, key):
        return int(row.iloc[0][key]) if len(row) else 0

    g0 = _get(row_ab, 'games')
    w0 = _get(row_ab, 'wins_agent0')  # wins when agent plays as player 0
    g1 = _get(row_ba, 'games')
    w1 = _get(row_ba, 'wins_agent1')  # wins when agent plays as player 1

    p0 = (w0 / g0 * 100.0) if g0 else float('nan')
    p1 = (w1 / g1 * 100.0) if g1 else float('nan')

    merged_games = g0 + g1
    merged_wins = w0 + w1
    merged = (merged_wins / merged_games * 100.0) if merged_games else float('nan')

    return p0, p1, merged

_results = {}
_export_rows = []

for a in _top5_agents:
    opponents = [o for o in AGENTS if o != a]
    records = []
    for o in opponents:
        p0, p1, merged = _pair_stats(a, o, tournament_df)
        min_role = np.nanmin([p0, p1]) if (not math.isnan(p0) or not math.isnan(p1)) else float('nan')
        records.append({
            'opponent': o,
            'merged_win_percent': merged,
            'p0_win_percent': p0,
            'p1_win_percent': p1,
            'min_role_win_percent': min_role,
        })

    rec_df = pd.DataFrame(records)

    # 1) Worst 5 by merged win % (lowest first)
    worst5 = rec_df.sort_values('merged_win_percent', ascending=True).head(5)

    # 2) Additional adversaries where any role < 50% (and not already in worst5), sorted by worst-role ascending
    remaining = rec_df[~rec_df['opponent'].isin(worst5['opponent'])].copy()
    role_vuln = remaining[remaining['min_role_win_percent'] < 50].sort_values('min_role_win_percent', ascending=True)

    _results[a] = {
        'worst_merged': worst5,
        'role_vulnerabilities': role_vuln,
    }

    # Prepare export rows
    for _, r in worst5.iterrows():
        _export_rows.append({
            'agent': a,
            'opponent': r['opponent'],
            'type': 'merged',
            'merged_win_percent': r['merged_win_percent'],
            'p0_win_percent': r['p0_win_percent'],
            'p1_win_percent': r['p1_win_percent'],
            'min_role_win_percent': r['min_role_win_percent'],
        })
    for _, r in role_vuln.iterrows():
        _export_rows.append({
            'agent': a,
            'opponent': r['opponent'],
            'type': 'role<50',
            'merged_win_percent': r['merged_win_percent'],
            'p0_win_percent': r['p0_win_percent'],
            'p1_win_percent': r['p1_win_percent'],
            'min_role_win_percent': r['min_role_win_percent'],
        })

# Pretty print the summary
print("\n" + "="*100)
print("TOP-5 AGENTS: WORST ADVERSARIES")
print("="*100)
for a in _top5_agents:
    print(f"\nAgent: {a}")
    print("- Worst 5 by merged win% (lowest first):")
    w = _results[a]['worst_merged'][['opponent','merged_win_percent','p0_win_percent','p1_win_percent']].copy()
    print(w.to_string(index=False, formatters={
        'merged_win_percent': lambda x: f"{x:.2f}",
        'p0_win_percent': lambda x: f"{x:.2f}",
        'p1_win_percent': lambda x: f"{x:.2f}",
    }))

    rv = _results[a]['role_vulnerabilities'][['opponent','min_role_win_percent','p0_win_percent','p1_win_percent','merged_win_percent']].copy()
    if len(rv) > 0:
        print("\n- Additional role vulnerabilities (any role < 50%):")
        print(rv.to_string(index=False, formatters={
            'min_role_win_percent': lambda x: f"{x:.2f}",
            'p0_win_percent': lambda x: f"{x:.2f}",
            'p1_win_percent': lambda x: f"{x:.2f}",
            'merged_win_percent': lambda x: f"{x:.2f}",
        }))
    else:
        print("\n- Additional role vulnerabilities: None (<50%)")

# Optional: export details
_out = os.path.join(DATA_DIR, 'top5_worst_adversaries.csv')
pd.DataFrame(_export_rows).to_csv(_out, index=False)
print(f"\nExported details to: {_out}")



TOP-5 AGENTS: WORST ADVERSARIES

Agent: rl_ppo
- Worst 5 by merged win% (lowest first):
         opponent merged_win_percent p0_win_percent p1_win_percent
       alternator              50.00         100.00           0.00
         bayesian              50.00         100.00           0.00
random_aggressive              88.00         100.00          76.00
random_facerandom              89.50         100.00          79.00
           random              91.00         100.00          82.00

- Additional role vulnerabilities: None (<50%)

Agent: bayesian
- Worst 5 by merged win% (lowest first):
   opponent merged_win_percent p0_win_percent p1_win_percent
onesarewild              50.00         100.00           0.00
   maxcount              50.00         100.00           0.00
     rl_ppo              50.00         100.00           0.00
   safeface              50.00         100.00           0.00
     parity              62.00         100.00          24.00

- Additional role vulnerabilities (a

In [50]:
# Statistical Test: First-Mover Advantage Analysis
print("\n" + "="*100)
print("FIRST-MOVER ADVANTAGE: STATISTICAL ANALYSIS")
print("="*100)

# Aggregate wins across all agents by role
total_wins_as_starter = agent_df['wins_as_starter'].sum()
total_wins_as_second = agent_df['wins_as_second'].sum()
total_games_as_starter = (agent_df['games'] // 2).sum()  # Each agent plays half games as starter
total_games_as_second = (agent_df['games'] // 2).sum()

# Overall win rates by position
starter_win_rate = (total_wins_as_starter / total_games_as_starter * 100) if total_games_as_starter > 0 else 0
second_win_rate = (total_wins_as_second / total_games_as_second * 100) if total_games_as_second > 0 else 0
advantage = starter_win_rate - second_win_rate

print(f"\nAggregate Statistics:")
print(f"  Games as Player 0 (starter): {total_games_as_starter:,}")
print(f"  Wins as Player 0: {total_wins_as_starter:,}")
print(f"  Win rate as Player 0: {starter_win_rate:.3f}%")
print()
print(f"  Games as Player 1 (second): {total_games_as_second:,}")
print(f"  Wins as Player 1: {total_wins_as_second:,}")
print(f"  Win rate as Player 1: {second_win_rate:.3f}%")
print()
print(f"  Raw Advantage (P0 - P1): {advantage:+.3f}%")

# Binomial test: null hypothesis is that starting position doesn't matter (p=0.5)
# Alternative: two-sided test to detect any deviation from 50%
total_games = total_games_as_starter + total_games_as_second
total_wins_p0 = total_wins_as_starter
binomial_result = stats.binomtest(total_wins_p0, total_games, 0.5, alternative='two-sided')

print(f"\nStatistical Significance Test (Binomial Test):")
print(f"  Null Hypothesis: No position advantage (true win rate = 50%)")
print(f"  Observed P0 win rate: {(total_wins_p0 / total_games * 100):.3f}%")
print(f"  p-value: {binomial_result.pvalue:.6f}")

if binomial_result.pvalue < 0.001:
    sig_level = "highly significant (p < 0.001)"
elif binomial_result.pvalue < 0.01:
    sig_level = "very significant (p < 0.01)"
elif binomial_result.pvalue < 0.05:
    sig_level = "significant (p < 0.05)"
else:
    sig_level = "not significant (p >= 0.05)"

print(f"  Result: {sig_level}")

# 95% Confidence interval for the advantage
p_hat = total_wins_p0 / total_games
se = np.sqrt(p_hat * (1 - p_hat) / total_games)
ci_lower = (p_hat - 1.96 * se) * 100
ci_upper = (p_hat + 1.96 * se) * 100
ci_advantage_lower = ci_lower - 50
ci_advantage_upper = ci_upper - 50

print(f"\n95% Confidence Interval for P0 Win Rate:")
print(f"  [{ci_lower:.3f}%, {ci_upper:.3f}%]")
print(f"  Advantage CI: [{ci_advantage_lower:+.3f}%, {ci_advantage_upper:+.3f}%]")

# Effect size (Cohen's h for proportions)
# h = 2 * arcsin(sqrt(p1)) - 2 * arcsin(sqrt(p2))
p1 = total_wins_as_starter / total_games_as_starter
p2 = total_wins_as_second / total_games_as_second
cohens_h = 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

print(f"\nEffect Size (Cohen's h):")
print(f"  h = {cohens_h:.4f}")
if abs(cohens_h) < 0.2:
    effect_desc = "negligible"
elif abs(cohens_h) < 0.5:
    effect_desc = "small"
elif abs(cohens_h) < 0.8:
    effect_desc = "medium"
else:
    effect_desc = "large"
print(f"  Interpretation: {effect_desc} effect")

# Interpretation
print(f"\n" + "-"*100)
print("INTERPRETATION:")
print("-"*100)

if binomial_result.pvalue < 0.05:
    direction = "advantage" if advantage > 0 else "disadvantage"
    print(f"✓ There IS a statistically significant first-mover {direction}.")
    print(f"  Player 0 (starting player) wins {abs(advantage):.2f}% more/less often than Player 1.")
    print(f"  This difference is {sig_level}.")
    print(f"  Effect size is {effect_desc}, meaning the advantage is {effect_desc} in practical terms.")
else:
    print(f"✗ There is NO statistically significant first-mover advantage.")
    print(f"  The observed difference ({advantage:+.3f}%) is likely due to chance.")
    print(f"  Starting position does not confer a meaningful advantage in Liar's Dice.")

print("="*100)



FIRST-MOVER ADVANTAGE: STATISTICAL ANALYSIS

Aggregate Statistics:
  Games as Player 0 (starter): 156,800
  Wins as Player 0: 78,937
  Win rate as Player 0: 50.342%

  Games as Player 1 (second): 156,800
  Wins as Player 1: 77,863
  Win rate as Player 1: 49.658%

  Raw Advantage (P0 - P1): +0.685%

Statistical Significance Test (Binomial Test):
  Null Hypothesis: No position advantage (true win rate = 50%)
  Observed P0 win rate: 25.171%
  p-value: 0.000000
  Result: highly significant (p < 0.001)

95% Confidence Interval for P0 Win Rate:
  [25.019%, 25.323%]
  Advantage CI: [-24.981%, -24.677%]

Effect Size (Cohen's h):
  h = 0.0137
  Interpretation: negligible effect

----------------------------------------------------------------------------------------------------
INTERPRETATION:
----------------------------------------------------------------------------------------------------
✓ There IS a statistically significant first-mover advantage.
  Player 0 (starting player) wins 0.68% 